In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os,numpy as np,pandas as pd,torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
from torchvision import datasets,transforms,models
from PIL import Image
from sklearn.model_selection import train_test_split
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_dir='/kaggle/input/competitions/2026task1/dataset-for-task2/dataset-for-task2'
train_dir=f'{data_dir}/train'
test_dir=f'{data_dir}/test'
# 训练集按文件夹名自动得到类别（注意:ImageFolder按字母序排列）
classes=sorted(os.listdir(train_dir))
print('5个类别:',classes)
# 输出: ['Black-grass', 'Common wheat', 'Loose Silky-bent', 'Scentless Mayweed', 'Sugar beet']

5 个类别: ['Black-grass', 'Common wheat', 'Loose Silky-bent', 'Scentless Mayweed', 'Sugar beet']


In [5]:
train_tf = transforms.Compose([transforms.Resize((224, 224)),transforms.RandomHorizontalFlip(),transforms.RandomRotation(15),transforms.ColorJitter(0.2, 0.2, 0.2),transforms.ToTensor(),transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
test_tf=transforms.Compose([transforms.Resize((224, 224)),transforms.ToTensor(),transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
train_ds=datasets.ImageFolder(train_dir, transform=train_tf)
#切出20%验证集
train_idx, val_idx=train_test_split(range(len(train_ds)),test_size=0.2,stratify=train_ds.targets,random_state=42)
train_dl=DataLoader(train_ds, batch_size=16,sampler=torch.utils.data.SubsetRandomSampler(train_idx))
val_dl=DataLoader(train_ds, batch_size=16,sampler=torch.utils.data.SubsetRandomSampler(val_idx))

In [7]:
class TestDataset(Dataset):
    def __init__(self, folder, transform):
        self.files=sorted(os.listdir(folder))      # 文件名即 ID
        self.folder=folder
        self.transform=transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self,i):
        img = Image.open(os.path.join(self.folder, self.files[i])).convert('RGB')
        return self.transform(img),self.files[i]
test_ds=TestDataset(test_dir,test_tf)
test_dl=DataLoader(test_ds,batch_size=16,shuffle=False)

In [8]:
model=models.resnet18(weights='IMAGENET1K_V1')
model.fc=nn.Linear(model.fc.in_features, len(classes))
model=model.to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(), lr=1e-3)
best_acc=0
for epoch in range(15):
    model.train()
    total_loss = 0
    for x,y in train_dl:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x),y)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    model.eval()
    correct=total=0
    with torch.no_grad():
        for x,y in val_dl:
            x,y=x.to(device),y.to(device)
            correct+=(model(x).argmax(1)==y).sum().item()
            total+=y.size(0)
    acc=correct/total
    print(f'Epoch{epoch+1:2d}|loss{total_loss/len(train_dl):.4f}|val_acc{acc:.4f}')
    if acc>best_acc:
        best_acc=acc
        torch.save(model.state_dict(),'best_model.pth')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 150MB/s] 


Epoch  1 | loss 1.1681 | val_acc 0.3200
Epoch  2 | loss 0.8356 | val_acc 0.6900
Epoch  3 | loss 0.6891 | val_acc 0.7600
Epoch  4 | loss 0.5348 | val_acc 0.7300
Epoch  5 | loss 0.6050 | val_acc 0.7900
Epoch  6 | loss 0.4556 | val_acc 0.7400
Epoch  7 | loss 0.5006 | val_acc 0.8400
Epoch  8 | loss 0.4243 | val_acc 0.7700
Epoch  9 | loss 0.3765 | val_acc 0.7300
Epoch 10 | loss 0.4480 | val_acc 0.6600
Epoch 11 | loss 0.4140 | val_acc 0.7200
Epoch 12 | loss 0.4209 | val_acc 0.7800
Epoch 13 | loss 0.2964 | val_acc 0.7400
Epoch 14 | loss 0.3630 | val_acc 0.7800
Epoch 15 | loss 0.3480 | val_acc 0.6800


In [9]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
pred_ids,pred_labels=[],[]
with torch.no_grad():
    for x,filenames in test_dl:
        x=x.to(device)
        preds=model(x).argmax(1).cpu()
        pred_ids.extend(filenames)
        pred_labels.extend([classes[p] for p in preds])
submission=pd.DataFrame({'ID':pred_ids,'Category':pred_labels})
submission.to_csv('submission.csv',index=False)
print(submission.head())

               ID          Category
0  abkieurfrr.png        Sugar beet
1  ablzkpojla.png        Sugar beet
2  acmbgjflct.png  Loose Silky-bent
3  afhcnytjtd.png  Loose Silky-bent
4  ahoyeyvosg.png       Black-grass
